In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.multioutput import RegressorChain
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score



In [2]:
df = pd.read_csv('preprocessed.csv')
df.set_index('Date', inplace=True)
cat = df.select_dtypes(include=['object']).columns.tolist()

x = df.drop(['Units Sold', 'Demand'], axis=1)
y = df[['Units Sold', 'Demand']]

x_train, x_test, y_train, y_test = train_test_split(
    x, y,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

C:\Users\kumar\AppData\Local\Temp\ipykernel_29940\3298373648.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat = df.select_dtypes(include=['object']).columns.tolist()


In [3]:
ct = ColumnTransformer(
    transformers=[
        ('encoder', OneHotEncoder(), cat)
    ],
    remainder='passthrough'
)

x_train = ct.fit_transform(x_train)
x_test = ct.transform(x_test)

x_train[np.isinf(x_train)] = np.nan
x_test[np.isinf(x_test)] = np.nan

imputer = SimpleImputer(strategy='median')

x_train = imputer.fit_transform(x_train)
x_test = imputer.transform(x_test)


In [4]:
import optuna
from sklearn.model_selection import cross_val_score
from sklearn.multioutput import RegressorChain
from xgboost import XGBRegressor

def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 500, 1000, step=50),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2),
        "max_depth": trial.suggest_int("max_depth", 3, 7),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 5),
        "gamma": trial.suggest_float("gamma", 0.0, 0.5, step=0.1),
        "subsample": trial.suggest_float("subsample", 0.8, 1.0, step=0.1),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "random_state": 42,
        "n_jobs": -1,
    }

    xgb = XGBRegressor(**params)
    regressor_chain = RegressorChain(xgb, order=[0, 1])

    scores = cross_val_score(
        regressor_chain, x_train, y_train, cv=5, scoring="r2"
    )
    return scores.mean()


study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=50) 



[I 2026-06-02 08:35:49,334] A new study created in memory with name: no-name-d732447f-fef2-4cf6-b4de-575f3741e296
[I 2026-06-02 08:36:36,068] Trial 0 finished with value: 0.802667160213649 and parameters: {'n_estimators': 700, 'learning_rate': 0.19063571821788408, 'max_depth': 6, 'min_child_weight': 3, 'gamma': 0.0, 'subsample': 0.8, 'colsample_bytree': 0.6232334448672797, 'reg_lambda': 2.9154431891537547, 'reg_alpha': 0.2537815508265665}. Best is trial 0 with value: 0.802667160213649.
[I 2026-06-02 08:38:13,508] Trial 1 finished with value: 0.6634865418327411 and parameters: {'n_estimators': 850, 'learning_rate': 0.013911053916202464, 'max_depth': 7, 'min_child_weight': 5, 'gamma': 0.1, 'subsample': 0.8, 'colsample_bytree': 0.6733618039413735, 'reg_lambda': 0.016480446427978974, 'reg_alpha': 0.12561043700013558}. Best is trial 0 with value: 0.802667160213649.
[I 2026-06-02 08:39:14,645] Trial 2 finished with value: 0.7605790213991253 and parameters: {'n_estimators': 700, 'learning_rat

In [5]:

print("\n--- Optimization Finished ---")
print(f"Best CV R2 Score: {study.best_value:.4f}")
print("Best Hyperparameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")


--- Optimization Finished ---
Best CV R2 Score: 0.8172
Best Hyperparameters:
  n_estimators: 850
  learning_rate: 0.15891737958931385
  max_depth: 6
  min_child_weight: 4
  gamma: 0.4
  subsample: 0.9
  colsample_bytree: 0.8835943165796013
  reg_lambda: 0.3287992672892712
  reg_alpha: 0.010284447825378754


In [6]:

best_xgb = XGBRegressor(**study.best_params, random_state=42)
final_chain = RegressorChain(best_xgb, order=[0,1])
final_chain.fit(x_train, y_train)

y_pred = final_chain.predict(x_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"RMSE: {np.sqrt(mse)}")
print(f"R2: {r2}")

RMSE: 18.044159774897295
R2: 0.8343563880195289


In [7]:
import joblib
joblib.dump(final_chain, 'xgb_model.pkl')

joblib.dump(ct,'ColumnTransformer.pkl')

joblib.dump(imputer,"imputer.pkl")

['imputer.pkl']